# MongoDB and NoSQL document stores
* lesson 13 and lesson 40 both modeled our data **relationally**: fixed columns per table, a schema you must `CREATE TABLE` before you can insert a single row, and separate tables glued back together with `JOIN` (recap the `authors`/`books` example from both of those lessons)
* **NoSQL** ("not only SQL") is an umbrella term for databases that DONT work that way - there are several different "shapes" of NoSQL database (we'll list the other shapes at the end of this lesson), and **MongoDB** is by far the most popular example of one particular shape: the **document store**
* in a document store, we dont insert "rows into a table with fixed columns" - we insert **documents**, which are just JSON-like nested structures (in python: dicts, with lists and nested dicts inside), and we dont have to declare a schema up front at all
* instead of `JOIN`ing separate tables together, the usual document-store approach is to **embed** related data directly inside one document in the first place - though referencing (and even an approximate `JOIN`) is still possible, and we'll cover both approaches in depth below, because *choosing* between them is really the core skill this lesson teaches
* **be upfront about our setup**: there is no real MongoDB server running in this training environment. instead we use `mongomock`, a widely-used, purpose-built python library that mimics `pymongo`'s (MongoDB's official python driver) exact client API in memory, specifically designed for testing/teaching code that talks to MongoDB without needing a real server. every single method call we teach in this lesson (`insert_one`, `find`, `update_one`, `aggregate`, ...) works completely identically against a real MongoDB server or a MongoDB Atlas (the popular managed cloud version) connection - the ONLY line that would ever need to change is the very first one, where we construct the client (`mongomock.MongoClient()` here, vs. `pymongo.MongoClient("mongodb://localhost:27017/")` or an Atlas connection string in real life). to be fully honest: mongomock is very high-fidelity but doesnt implement literally every obscure edge case or aggregation operator a real server has - but it covers everything we need here faithfully, and its a completely legitimate, widely-used tool for exactly this kind of offline testing/teaching

In [1]:
import mongomock  # the in-memory pymongo-API-compatible mock we use for this whole lesson
import pymongo  # the real official driver - we only use this for a couple of type/exception references, no server needed for that

# this ONE line is the only thing that would differ from real usage -
# mongomock.MongoClient() behaves just like pymongo.MongoClient() for everything we do below
client = mongomock.MongoClient()

# a mongodb "database" is created lazily just by naming it - no CREATE DATABASE statement needed anywhere
db = client["l45_training"]

print("connected to (mock) mongodb, database name:", db.name)
print("driver we are pretending to talk to real mongo with:", pymongo.__name__, pymongo.version)

connected to (mock) mongodb, database name: l45_training
driver we are pretending to talk to real mongo with: pymongo 4.18.2


## basic terminology: mapping mongodb concepts onto sql concepts
* it helps a LOT to have a rough dictionary between the relational world (lesson 13/40) and the mongodb world before we touch any code

| MongoDB term | roughly the SQL equivalent | but... |
|---|---|---|
| **database** | database | same idea, a named container for everything else |
| **collection** | table | but a collection has NO fixed set of columns at all |
| **document** | row | but each document can have a COMPLETELY different set of fields from its neighbors in the same collection! |
| **field** | column | but fields can be nested (a field can itself hold a whole sub-document, or a list) |
| `_id` | primary key | mongodb auto-generates a unique `_id` (an `ObjectId`) for every document if we dont supply one ourselves |

* that "documents in the same collection can differ completely" point is the single biggest mental shift coming from SQL - in `books` (lesson 13), EVERY row has exactly the same columns (`id`, `title`, `year`, `author_id`), full stop, the database wont even let you insert a row missing a required column. in a mongodb `books` collection, one document could have a `series` field and the next one simply doesnt have that key at all - theres no schema to violate

In [2]:
books = db["books"]  # grabbing (creating, if it doesnt exist yet) a collection - just like grabbing a dict key, no CREATE TABLE needed

# lets prove the "no fixed schema" point immediately: two documents, genuinely different shapes
books.insert_one({
    "title": "Mort",
    "genre": "fantasy",
    "rating": 4.6,
    "series": "Discworld",          # this field only makes sense for series books
    "series_number": 4,              # ditto
})
books.insert_one({
    "title": "Foundation",
    "genre": "sci-fi",
    "rating": 4.4,
    # no "series"/"series_number" fields at all here - and that is completely fine, no error, no NULL columns needed
})

# a real sql table would need "series"/"series_number" columns on EVERY row (as NULL where unused) - mongodb just omits the key
for doc in books.find():
    print(doc)

{'title': 'Mort', 'genre': 'fantasy', 'rating': 4.6, 'series': 'Discworld', 'series_number': 4, '_id': ObjectId('6ab7b1f2ad64532e2fe901c5')}
{'title': 'Foundation', 'genre': 'sci-fi', 'rating': 4.4, '_id': ObjectId('6ab7b1f2ad64532e2fe901c6')}


## CRUD basics: insert, find, update, delete
* the four operations map directly onto SQL's INSERT / SELECT / UPDATE / DELETE, just with different method names and python dicts instead of SQL strings
* **insert**: `insert_one(document)` adds a single document, `insert_many([...])` adds several at once (like `executemany` in lesson 13)
* **find**: `find_one(filter)` returns the first matching document (or `None`), `find(filter)` returns a cursor we can iterate over all matches - the `filter` is just a plain python dict, e.g. `{"genre": "sci-fi"}`. this LOOKS a lot like a SQL `WHERE genre = 'sci-fi'`, but its really important to understand its NOT a query language at all - its literally just a python dict describing "fields that must match this value", theres no separate syntax to learn for the basic case
* **update**: `update_one(filter, changes)` / `update_many(filter, changes)` - the `changes` argument uses `$`-prefixed **update operators**, most commonly `$set` (set/overwrite specific fields, leaving everything else untouched) - the `$` prefix is mongodb's general convention for "this key names an operation, not a literal field name"
* **delete**: `delete_one(filter)` / `delete_many(filter)` - same filter-dict idea as find/update

In [3]:
# insert_many - bulk insert, conceptually identical to lesson 13's executemany()
books.insert_many([
    {"title": "The Left Hand of Darkness", "genre": "sci-fi", "rating": 4.7, "year": 1969},
    {"title": "I, Robot", "genre": "sci-fi", "rating": 4.3, "year": 1950},
    {"title": "Guards! Guards!", "genre": "fantasy", "rating": 4.5, "series": "Discworld", "series_number": 8},
])

# find_one - just the FIRST matching document, or None if nothing matches
one_scifi_book = books.find_one({"genre": "sci-fi"})  # the filter dict looks like a WHERE clause but is plain python
print("find_one result:", one_scifi_book)

# find - returns a Cursor (lazy, iterable) of EVERY matching document, like fetchall()/iterating a cursor in lesson 13
print("\nall sci-fi books:")
for book in books.find({"genre": "sci-fi"}):  # again: just a dict, no query string anywhere
    print(" -", book["title"], f"({book.get('year', 'year unknown')})")  # .get() is handy since not every doc has "year"

# find() also supports an optional PROJECTION (2nd argument) to only get back the fields we actually want
print("\ntitles only, projected:")
for book in books.find({}, {"_id": 0, "title": 1}):  # {} filter = match everything, projection: hide _id, keep only title
    print(" -", book)

# update_one with $set - changes ONLY the given fields on the FIRST matching document, leaves everything else alone
result = books.update_one(
    {"title": "I, Robot"},                 # filter: which document to touch
    {"$set": {"rating": 4.4}},             # $set is the update operator: "set these fields to these values"
)
print("\nmatched:", result.matched_count, "| modified:", result.modified_count)  # like .rowcount in lesson 13

# update_many with $set - bumps a "read_count" field (adding a BRAND NEW field!) on every fantasy book at once
books.update_many({"genre": "fantasy"}, {"$set": {"shelf": "fantasy-shelf-3"}})
print("fantasy books now have a 'shelf' field:")
for book in books.find({"genre": "fantasy"}, {"_id": 0, "title": 1, "shelf": 1}):
    print(" -", book)

# delete_one - removes the FIRST document matching the filter
del_result = books.delete_one({"title": "Foundation"})
print("\ndeleted count:", del_result.deleted_count)
print("remaining book count:", books.count_documents({}))  # count_documents({}) with an empty filter = "count everything"

find_one result: {'title': 'Foundation', 'genre': 'sci-fi', 'rating': 4.4, '_id': ObjectId('6ab7b1f2ad64532e2fe901c6')}

all sci-fi books:
 - Foundation (year unknown)
 - The Left Hand of Darkness (1969)
 - I, Robot (1950)

titles only, projected:
 - {'title': 'Mort'}
 - {'title': 'Foundation'}
 - {'title': 'The Left Hand of Darkness'}
 - {'title': 'I, Robot'}
 - {'title': 'Guards! Guards!'}

matched: 1 | modified: 1
fantasy books now have a 'shelf' field:
 - {'title': 'Mort', 'shelf': 'fantasy-shelf-3'}
 - {'title': 'Guards! Guards!', 'shelf': 'fantasy-shelf-3'}

deleted count: 1
remaining book count: 4


## the big modeling decision: embedding vs. referencing
* this is the conceptual heart of this whole lesson. in sql (lesson 13) there was really only one way to connect authors and books: two separate tables, linked by a foreign key, combined again with `JOIN` when we need both together
* in mongodb, we have a genuine CHOICE for every "this thing belongs to/contains that thing" relationship, and picking the right one for the situation is a real, important design skill:
  * **embedding** - put the "child" data directly INSIDE the parent document. good when the child is basically always read together with its parent, and rarely (or never) needs to be queried completely on its own
  * **referencing** - keep the child in its OWN separate collection, and store only an id (`author_id`, exactly like lesson 13's foreign key idea) inside the parent document, pointing at it. good when the same child is shared/referenced by MANY parents, or when the child collection gets queried independently a lot
* the tradeoff, stated plainly:
  * embedding avoids a second lookup entirely (one single `find()` gets you everything, similar in spirit to how embedding avoids needing a JOIN at all) - but it DUPLICATES the child's data into every parent that embeds it, so if the child's data changes, you potentially have to update it in many places to keep everything in sync
  * referencing avoids ANY duplication (the author's data lives in exactly one place) - but reading "a book with its author" now needs a second query/lookup, since mongodb has no true server-side JOIN the way sql does (a `$lookup` aggregation stage, shown further below, can approximate one though)
* lets build the SAME `authors`/`books` example from lesson 13, but modeled BOTH ways, so we can compare directly

### option A: embedding (author lives inside each book document)
* good fit here IF we mostly only ever look up a book together with "its" author info, and authors data barely ever changes on its own

In [4]:
books_embedded = db["books_embedded"]  # a fresh collection just for this embedding demo

# each book document carries its FULL author info directly inside an "author" sub-document (nested dict)
books_embedded.insert_many([
    {
        "title": "The Left Hand of Darkness",
        "year": 1969,
        "author": {"name": "Ursula K. Le Guin", "birth_year": 1929},  # embedded sub-document, no separate lookup needed
    },
    {
        "title": "The Dispossessed",
        "year": 1974,
        "author": {"name": "Ursula K. Le Guin", "birth_year": 1929},  # NOTE: le guin's data is now duplicated here too!
    },
    {
        "title": "Foundation",
        "year": 1951,
        "author": {"name": "Isaac Asimov", "birth_year": 1920},
    },
])

# reading a book AND its author info needs just ONE single find() - no second query at all, nice and fast
for book in books_embedded.find({}, {"_id": 0}):
    print(f"{book['title']} ({book['year']}) - by {book['author']['name']}, born {book['author']['birth_year']}")

# the downside in action: if we ever learn le guin's birth year was recorded wrong, we now have to
# fix it in EVERY document that embeds her - here thats 2 documents, but imagine 50 books by the same author
print("\nfixing a typo in le guin's birth year - has to happen in EVERY embedded copy:")
books_embedded.update_many(
    {"author.name": "Ursula K. Le Guin"},          # dot-notation lets us filter on a field INSIDE a nested sub-document
    {"$set": {"author.birth_year": 1929}},          # and $set can target a nested field the same way
)
print("done - but notice we had to touch every matching document, not just one 'authors' row")

The Left Hand of Darkness (1969) - by Ursula K. Le Guin, born 1929
The Dispossessed (1974) - by Ursula K. Le Guin, born 1929
Foundation (1951) - by Isaac Asimov, born 1920

fixing a typo in le guin's birth year - has to happen in EVERY embedded copy:
done - but notice we had to touch every matching document, not just one 'authors' row


### option B: referencing (separate authors and books collections, linked by an id)
* good fit here IF the same author writes many books and we dont want to duplicate/maintain their data in dozens of places, or if we need to query authors independently a lot (e.g. "list every author born before 1930" without caring about books at all)
* this is structurally identical to lesson 13's `authors`/`books` tables with a foreign key - just without an enforced schema

In [5]:
authors_ref = db["authors_ref"]   # separate collection, one document per author - the "one source of truth"
books_ref = db["books_ref"]        # separate collection, one document per book, referencing an author by id

# insert_one returns an InsertOneResult whose .inserted_id is the auto-generated (or provided) _id - like lastrowid in lesson 13
le_guin_id = authors_ref.insert_one({"name": "Ursula K. Le Guin", "birth_year": 1929}).inserted_id
asimov_id = authors_ref.insert_one({"name": "Isaac Asimov", "birth_year": 1920}).inserted_id

print("le guin's generated _id:", le_guin_id, "| type:", type(le_guin_id).__name__)  # an ObjectId, mongodb's default id type

# each book stores just the author's _id - exactly like books.author_id in lesson 13's sqlite schema
books_ref.insert_many([
    {"title": "The Left Hand of Darkness", "year": 1969, "author_id": le_guin_id},
    {"title": "The Dispossessed", "year": 1974, "author_id": le_guin_id},   # le guin's DATA is stored only ONCE, here just referenced twice
    {"title": "Foundation", "year": 1951, "author_id": asimov_id},
])

# now fixing le guin's birth year needs exactly ONE update, in ONE document, no matter how many books she has
authors_ref.update_one({"_id": le_guin_id}, {"$set": {"birth_year": 1929}})
print("fixed le guin's birth year in exactly one place (the authors_ref collection)")

# but reading "a book with its author name" now needs TWO queries by hand, since there is no automatic JOIN:
for book in books_ref.find({}, {"_id": 0}):
    author = authors_ref.find_one({"_id": book["author_id"]}, {"_id": 0, "name": 1})  # the "second lookup" the tradeoff talks about
    print(f"{book['title']} ({book['year']}) - by {author['name']}")

le guin's generated _id: 6ab7b1f2ad64532e2fe901cd | type: ObjectId
fixed le guin's birth year in exactly one place (the authors_ref collection)
The Left Hand of Darkness (1969) - by Ursula K. Le Guin
The Dispossessed (1974) - by Ursula K. Le Guin
Foundation (1951) - by Isaac Asimov


## indexes: making lookups fast
* same underlying idea as an sql index (recap lesson 13/40) - without an index, a query filtering on a field has to scan through EVERY single document in the collection one by one to check if it matches ("a full collection scan"), which gets slow fast as a collection grows
* `collection.create_index("field_name")` builds a sorted structure mongodb can binary-search instead of scanning, making lookups filtered on (or sorted by) that field dramatically faster on bigger collections
* the cost: an index takes up extra storage and makes writes (insert/update/delete) very slightly slower, since the index has to be kept up to date too - so we only add indexes on fields we actually query/sort by often, not on everything

In [6]:
# without an index, filtering books_ref by author_id has to scan every single document to check author_id == le_guin_id
# thats totally fine for our tiny 3-document demo collection, but would get genuinely slow with millions of books

index_name = books_ref.create_index("author_id")  # builds an index on the author_id field specifically
print("created index:", index_name)

# create_index also works on nested fields, using the same dot-notation we saw with update_many above
authors_index_name = books_embedded.create_index("author.name")  # speeds up filtering embedded docs by author name
print("created index:", authors_index_name)

# we can list every index a collection currently has, to see what mongodb will actually use to speed up queries
print("\nindexes on books_ref:")
for index_info in books_ref.list_indexes():
    print(" -", index_info["name"], "on key(s):", dict(index_info["key"]))

# this query (find all books by le_guin_id) is EXACTLY the query pattern the author_id index above speeds up
matching = list(books_ref.find({"author_id": le_guin_id}))
print(f"\nfound {len(matching)} books by that author_id - this exact filter pattern is what the index accelerates")

created index: author_id_1
created index: author.name_1

indexes on books_ref:
 - _id_ on key(s): {'_id': 1}
 - author_id_1 on key(s): {'author_id': 1}

found 2 books by that author_id - this exact filter pattern is what the index accelerates


## the aggregation pipeline: mongodb's answer to GROUP BY and JOIN
* sql has `GROUP BY` (with aggregate functions like `AVG`/`SUM`) and `JOIN` built right into the query language itself (lesson 13). mongodb's equivalent is the **aggregation pipeline** - a LIST of processing "stages", where each stage takes the documents coming out of the previous stage and transforms them further, like a series of python list-comprehensions chained together
* `collection.aggregate([stage1, stage2, ...])` runs the whole pipeline and returns a cursor of the final results
* the stages we use here:
  * `$match` - filters documents, exactly like `$WHERE` /a `find()` filter (often used as the FIRST stage, to shrink the data early)
  * `$group` - groups documents by some field and computes an aggregate PER group, using accumulator operators like `$sum` (total count/sum), `$avg` (average) - this is directly `GROUP BY` + `AVG()`/`SUM()` from sql
  * `$lookup` - pulls in matching documents from ANOTHER collection based on a field match, approximating a sql `JOIN` (though its still a separate lookup step under the hood, not a true relational join)

In [7]:
# lets add a few more books with genres+ratings so the grouping below has more to work with
books.insert_many([
    {"title": "Mort", "genre": "fantasy", "rating": 4.6},
    {"title": "The Colour of Magic", "genre": "fantasy", "rating": 4.1},
])

# $match ~ WHERE, $group ~ GROUP BY with an accumulator ~ AVG()/COUNT() - computing average rating PER genre
pipeline_avg_rating = [
    {"$match": {"rating": {"$exists": True}}},           # only consider documents that actually HAVE a rating field at all
    {"$group": {
        "_id": "$genre",                                   # group by the "genre" field - this becomes each result's _id
        "average_rating": {"$avg": "$rating"},              # accumulator: average of "rating" within each group
        "book_count": {"$sum": 1},                           # accumulator: $sum 1 per document = a simple count, like COUNT(*)
    }},
    {"$sort": {"average_rating": -1}},                       # extra stage: sort results, highest average rating first
]

print("average rating per genre:")
for group in books.aggregate(pipeline_avg_rating):
    print(f" - {group['_id']}: avg {group['average_rating']:.2f} across {group['book_count']} book(s)")

# $lookup ~ approximates a JOIN - pulling each book's referenced author document from the authors_ref collection
pipeline_join_authors = [
    {"$lookup": {
        "from": "authors_ref",           # the OTHER collection to pull matching documents from
        "localField": "author_id",        # the field on THIS (books_ref) document to match...
        "foreignField": "_id",             # ...against THIS field on the other (authors_ref) document
        "as": "author_docs",                # $lookup always returns a LIST, stored under this new field name
    }},
    {"$unwind": "$author_docs"},           # author_docs is a list with exactly one match here - unwind flattens it back to a single sub-document
    {"$project": {                          # $project ~ SELECT specific columns: reshape the final output
        "_id": 0,
        "title": 1,
        "author_name": "$author_docs.name",  # pull just the name out of the (formerly nested) looked-up author document
    }},
]

print("\nbooks joined with their referenced author (via $lookup):")
for row in books_ref.aggregate(pipeline_join_authors):
    print(" -", row)

average rating per genre:
 - sci-fi: avg 4.55 across 2 book(s)
 - fantasy: avg 4.45 across 4 book(s)

books joined with their referenced author (via $lookup):
 - {'title': 'The Left Hand of Darkness', 'author_name': 'Ursula K. Le Guin'}
 - {'title': 'The Dispossessed', 'author_name': 'Ursula K. Le Guin'}
 - {'title': 'Foundation', 'author_name': 'Isaac Asimov'}


## quick comparison to other languages
* `pymongo`'s api (and, by extension, everything we just did with `mongomock` above) is very close to MongoDB's official drivers for basically every other mainstream language - the method NAMES differ a bit per language convention, but the shape of everything is nearly identical
* Java has the official MongoDB Java driver, Node.js has the official `mongodb` npm package (and, layered on top of it, the very popular `mongoose` ODM, which adds OPTIONAL schema validation/structure back into mongodb - conceptually similar to how SQLAlchemy (lesson 40) sits on top of raw SQL/sqlite), C# has `MongoDB.Driver`, Go has `mongo-go-driver`
* the really important point: the actual QUERY documents and aggregation pipeline stages (the JSON-shaped filters like `{"genre": "sci-fi"}`, or pipeline stages like `$match`/`$group`/`$lookup`) are near-identical across every single one of these languages' drivers - because under the hood they all just get serialized into the same wire format and sent to the exact same mongodb server the exact same way. learning the QUERY language here transfers almost 1:1 to any other language's mongodb driver, much like SQL itself transfers across sql database engines
* for completeness: document stores are just ONE of several NoSQL "shapes" - **key-value stores** (Redis, recap lesson 44) store simple key -> value pairs, no structure inside the value required; **wide-column stores** (Cassandra, DynamoDB) are optimized for huge amounts of data spread across many machines, organized in flexible "column families" per row; **graph databases** (Neo4j) model data as nodes and the RELATIONSHIPS between them directly, which shines for highly interconnected data like social networks or recommendation engines, where "friends of friends of friends" queries would be painfully slow with JOINs in sql. each shape is optimized for a different access pattern - theres no single "best" database, only the one that fits how YOU need to read and write your data
* lesson 46 will directly put sql (relational) and nosql (document, in particular) modeling side by side and help build intuition for which one to reach for, for a given real-world problem

## Exercises
1. insert 3 book documents into a fresh collection where the documents have genuinely different sets of fields (e.g. only one of them has a `"series"` field) - then run a `find()` query and show it works fine despite the schema differences between the documents
2. write an `update_many()` call that adds a brand new field to ONLY the documents matching a filter of your choice, using `$set`
3. design your own small pair of referencing collections (like `books_ref`/`authors_ref` above, but pick your own theme, e.g. movies/directors or students/schools) and write a `$lookup` aggregation pipeline that joins them together
4. create an index on a field of your choice on any collection from this lesson, and in a comment explain what query pattern (which filter/sort) it would actually speed up

In [8]:
# TODO: solve exercise 1 here (insert 3 documents with different field sets, then find() to prove it works)


# TODO: solve exercise 2 here (update_many with $set, adding a new field to matching documents only)


# TODO: solve exercise 3 here (your own referencing collections + a $lookup aggregation)


# TODO: solve exercise 4 here (create_index + a comment explaining what query pattern it speeds up)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [9]:
# sample solution 1 - a fresh collection, 3 documents with genuinely different fields
gadgets = db["gadgets"]
gadgets.insert_many([
    {"name": "laptop", "price": 1200, "warranty_years": 2},               # no "wireless" field
    {"name": "mouse", "price": 25, "wireless": True},                      # no "warranty_years" field
    {"name": "keyboard", "price": 60, "wireless": False, "layout": "ansi"}, # has a field the others dont even have
])
print("exercise 1 - all gadgets despite different shapes:")
for gadget in gadgets.find({}, {"_id": 0}):
    print(" -", gadget)  # find() works completely fine, it never cared about a fixed schema in the first place

# sample solution 2 - update_many with $set, only touching documents matching a filter
gadgets.update_many(
    {"price": {"$lt": 100}},                     # filter: only cheaper gadgets (price under 100)
    {"$set": {"budget_friendly": True}},          # $set adds this brand new field to just those matching documents
)
print("\nexercise 2 - only cheap gadgets got the new field:")
for gadget in gadgets.find({}, {"_id": 0, "name": 1, "price": 1, "budget_friendly": 1}):
    print(" -", gadget)

# sample solution 3 - our own referencing pair: students referencing a school by id
schools = db["schools"]
students = db["students"]
hogwarts_id = schools.insert_one({"name": "Hogwarts"}).inserted_id
students.insert_many([
    {"name": "Harry", "school_id": hogwarts_id},
    {"name": "Hermione", "school_id": hogwarts_id},
])
join_pipeline = [
    {"$lookup": {"from": "schools", "localField": "school_id", "foreignField": "_id", "as": "school_docs"}},
    {"$unwind": "$school_docs"},  # school_docs is a one-element list per student here, flatten it to a single doc
    {"$project": {"_id": 0, "name": 1, "school_name": "$school_docs.name"}},
]
print("\nexercise 3 - students joined with their school via $lookup:")
for row in students.aggregate(join_pipeline):
    print(" -", row)

# sample solution 4 - an index, with a comment explaining exactly what it speeds up
gadgets.create_index("price")
# this index speeds up any query that FILTERS or SORTS by "price" - e.g. find({"price": {"$lt": 100}}) (used just above!)
# or find().sort("price", 1) - without it, mongodb would have to scan every single gadget document to check its price,
# with it, mongodb can jump straight to the relevant range instead, which matters a lot once the collection is big
print("\nexercise 4 - index created on 'price':", list(gadgets.list_indexes())[-1]["name"])

exercise 1 - all gadgets despite different shapes:
 - {'name': 'laptop', 'price': 1200, 'warranty_years': 2}
 - {'name': 'mouse', 'price': 25, 'wireless': True}
 - {'name': 'keyboard', 'price': 60, 'wireless': False, 'layout': 'ansi'}

exercise 2 - only cheap gadgets got the new field:
 - {'name': 'laptop', 'price': 1200}
 - {'name': 'mouse', 'price': 25, 'budget_friendly': True}
 - {'name': 'keyboard', 'price': 60, 'budget_friendly': True}

exercise 3 - students joined with their school via $lookup:
 - {'name': 'Harry', 'school_name': 'Hogwarts'}
 - {'name': 'Hermione', 'school_name': 'Hogwarts'}

exercise 4 - index created on 'price': price_1


* congratulation you finished this lesson

## what we learned
* what a NoSQL document store is, and how mongodb's terms (database/collection/document/field) map onto sql's (database/table/row/column) - with the crucial difference that documents in the same collection dont need a fixed, shared set of fields
* CRUD basics with pymongo's api: `insert_one`/`insert_many`, `find_one`/`find` with plain-dict filters (and optional projections), `update_one`/`update_many` with the `$set` update operator, `delete_one`/`delete_many`
* the central modeling decision in document databases: **embedding** related data directly inside a document (fast single reads, but duplicates data across documents) vs. **referencing** it in a separate collection by id (no duplication, but needs a second lookup, since mongodb has no true server-side JOIN)
* `create_index()` and why an index matters - the same underlying idea as an sql index, avoiding a full collection scan on every filtered/sorted query
* the aggregation pipeline (`$match`, `$group` with accumulators like `$avg`/`$sum`, `$lookup` to approximate a JOIN across collections) as mongodb's equivalent of sql's `WHERE`/`GROUP BY`/`JOIN`
* that `mongomock` is a faithful, in-memory stand-in for `pymongo`'s real api - everything we wrote here works unchanged against a real MongoDB server or Atlas, aside from the one client-construction line
* that document stores are just one of several NoSQL "shapes" (alongside key-value stores like redis, wide-column stores like cassandra/dynamodb, and graph databases like neo4j), each suited to a different access pattern